In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
import seaborn as sns # For heatmaps
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, train_test_split, cross_validate
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [2]:
df = pd.read_csv('ChEMBL_amines_12C_morgan_fingerprints.csv')

In [3]:
df

,ChEMBL ID,Name,Molecular Weight,CX Acidic pKa,CX Basic pKa,Aromatic Rings,Molecular Species,Molecular Formula,Smiles,Inchi Key,Amine Class,Morgan_Fingerprint
0,CHEMBL4297305,NB-001,264.33,NaN,10.25,2.0,BASE,C12H20N6O,Nc1ncnc2c1ncn2CCNCCCCCO,CVPTTZZCRDVGSU-UHFFFAOYSA-N,secondary,0000000000000000000001000000000000000000000000...
1,CHEMBL1574,PHENTERMINE,149.24,NaN,10.25,1.0,BASE,C10H15N,CC(C)(N)Cc1ccccc1,DHHVAGZRUROJKS-UHFFFAOYSA-N,primary,0000000000000000000000000000000000000000000000...
2,CHEMBL5394915,NaN,258.32,12.58,8.85,0.0,BASE,C12H22N2O4,COC(=O)[C@@H]1CCCN1C(=O)[C@@H](O)[C@H](N)C(C)C,UCLCHJWFVUTOCB-AEJSXWLSSA-N,primary,0100000000000000000000000000000000000000001000...
3,CHEMBL1546,HYDROXYAMPHETAMINE,151.21,10.48,9.80,1.0,BASE,C9H13NO,CC(N)Cc1ccc(O)cc1,GIKNHHRFLCDOEU-UHFFFAOYSA-N,primary,0100000000000000000000000000000000000000000000...
4,CHEMBL2105671,AFEGOSTAT TARTRATE,297.26,13.52,8.80,0.0,BASE,C10H19NO9,O=C(O)C(O)C(O)C(=O)O.OC[C@H]1CNC[C@@H](O)[C@@H]1O,ULBPPCHRAVUQMC-RWOHWRPJSA-N,secondary,0100000000000000000000000000000000000000000000...
...,...,...,...,...,...,...,...,...,...,...,...,...
5862,CHEMBL684,DIETHYLCARBAMAZINE,199.30,NaN,6.90,0.0,NEUTRAL,C10H21N3O,CCN(CC)C(=O)N1CCN(C)CC1,RCKMWOKWVGPNJF-UHFFFAOYSA-N,tertiary,0000000000000010000000000000000000000000000000...
5863,CHEMBL1086997,LUCERASTAT,219.28,12.90,8.49,0.0,NEUTRAL,C10H21NO4,CCCCN1C[C@H](O)[C@@H](O)[C@@H](O)[C@H]1CO,UQRORFVVSGFNRO-XFWSIPNHSA-N,tertiary,0000000000000000000000000000000000000000000000...
5864,CHEMBL511099,BICIFADINE,173.26,NaN,10.60,1.0,BASE,C12H15N,Cc1ccc(C23CNCC2C3)cc1,OFYVIGTWSQPCLF-UHFFFAOYSA-N,secondary,0000000000000000000000000000000000000000000000...
5865,CHEMBL358040,NORFENEFRINE,153.18,9.56,8.91,1.0,BASE,C8H11NO2,NCC(O)c1cccc(O)c1,LRCXRAABFLIVAI-UHFFFAOYSA-N,primary,0100000000000000000000000000000000000000000000...


In [4]:
# function to canonicalize a SMILES string
def canonicalize_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        return Chem.MolToSmiles(mol, canonical=True)
    return None  # or smi if you want to keep the original on failure

# Apply to the 'smiles' column of your DataFrame
df['canonical_smiles'] = df['Smiles'].apply(canonicalize_smiles)

In [5]:
df1 = df[['ChEMBL ID', 'CX Basic pKa', 'canonical_smiles']]

In [6]:
df1

,ChEMBL ID,CX Basic pKa,canonical_smiles
0,CHEMBL4297305,10.25,Nc1ncnc2c1ncn2CCNCCCCCO
1,CHEMBL1574,10.25,CC(C)(N)Cc1ccccc1
2,CHEMBL5394915,8.85,COC(=O)[C@@H]1CCCN1C(=O)[C@@H](O)[C@H](N)C(C)C
3,CHEMBL1546,9.80,CC(N)Cc1ccc(O)cc1
4,CHEMBL2105671,8.80,O=C(O)C(O)C(O)C(=O)O.OC[C@H]1CNC[C@@H](O)[C@@H]1O
...,...,...,...
5862,CHEMBL684,6.90,CCN(CC)C(=O)N1CCN(C)CC1
5863,CHEMBL1086997,8.49,CCCCN1C[C@H](O)[C@@H](O)[C@@H](O)[C@H]1CO
5864,CHEMBL511099,10.60,Cc1ccc(C23CNCC2C3)cc1
5865,CHEMBL358040,8.91,NCC(O)c1cccc(O)c1


In [7]:
# ---------------------------
# Compute Morgan Fingerprints
# ---------------------------
def smiles_to_morgan_fp(smi, radius=2, nBits=1024):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
        return list(fp)
    return [0] * nBits

df1['morgan_fp'] = df1['canonical_smiles'].apply(smiles_to_morgan_fp)

/tmp/ipykernel_16924/1448783578.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['morgan_fp'] = df1['canonical_smiles'].apply(smiles_to_morgan_fp)


In [8]:
# ---------------------------
# Prepare Features and Labels
# ---------------------------
X_list = pd.DataFrame(df1['morgan_fp'].tolist())

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X_list)
y = df1['CX Basic pKa']

In [9]:
# 1. First split: 90% train, 10% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42)
# Output shapes
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (5280, 1024) (5280,)
Test: (587, 1024) (587,)


In [10]:
# ---------- Search space for RF ----------
search_space = {
    "n_estimators": (100, 1000),
    "max_depth": (3, 50),
    "min_samples_split": (2, 50),
    "min_samples_leaf": (1, 20),
    # max_features can be float (fraction), or categorical:
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False],
}

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

opt_rf = BayesSearchCV(
    estimator=rf,
    search_spaces=search_space,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    return_train_score=True,
    verbose=0
)

In [11]:
opt_rf.fit(X_train, y_train)

BayesSearchCV(cv=5, estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
              n_jobs=-1, random_state=42, return_train_score=True,
              scoring='neg_mean_squared_error',
              search_spaces={'bootstrap': [True, False], 'max_depth': (3, 50),
                             'max_features': ['sqrt', 'log2'],
                             'min_samples_leaf': (1, 20),
                             'min_samples_split': (2, 50),
                             'n_estimators': (100, 1000)})

In [12]:
best_opt = opt_rf.best_estimator_
best_opt

RandomForestRegressor(bootstrap=False, max_depth=47, max_features='sqrt',
                      n_estimators=1000, n_jobs=-1, random_state=42)

In [13]:
# -------------------- CV metrics on training set (5-fold) --------------------
cv_scoring = {'r2': 'r2', 'mse': 'neg_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
cv_results = cross_validate(best_opt, X_train, y_train, cv=5, scoring=cv_scoring, n_jobs=-1, return_train_score=False)

cv_mse = -cv_results['test_mse'].mean()
cv_rmse = np.sqrt(cv_mse)
cv_mae = -cv_results['test_mae'].mean()
cv_r2 = cv_results['test_r2'].mean()

In [14]:
rf_pred = opt_rf.predict(X_test)
rf_pred_train = opt_rf.predict(X_train)

mse = mean_squared_error(y_test, rf_pred)
mae = mean_absolute_error(y_test, rf_pred)
r2 = r2_score(y_test, rf_pred)

mse_train = mean_squared_error(y_train, rf_pred_train)
mae_train = mean_absolute_error(y_train, rf_pred_train)
r2_train = r2_score(y_train, rf_pred_train)

In [16]:
print(mse)
print(mse_train)
print(cv_mse)

print(mae)
print(mae_train)
print(cv_mae)

print(r2)
print(r2_train)
print(cv_r2)

0.355685388381
0.0031681906338030595
0.4810809470861023
0.30970559549057336
0.03385348165909685
0.3674165978532132
0.810066960530638
0.9983441221579274
0.7485890614591497
